# 🧠 Fruit Classification with Transfer Learning (VGG16)

In this notebook I build an image classifier for fruit photographs by **reusing a network someone
else already trained**. Rather than learning to see from scratch, I take VGG16 — trained on ImageNet's
1.2 million images across 1000 categories — strip off its classifier head, and train a new head on my
own fruit dataset.

Then I go one step further and **fine-tune** the top of the borrowed network so it adapts to
fruit-specific detail rather than generic ImageNet features.

## 📋 Overview

Training a competitive image classifier from random initialisation needs enormous amounts of data and
compute. Transfer learning sidesteps both. The insight is that the *early* layers of a vision network
learn features that are almost universal — edges, corners, colour gradients, textures — and those
transfer to any image task. Only the *late* layers are specific to the original categories.

The engineering parallel is direct: I don't redesign an RF front-end from first principles for every
new product. The low-noise amplifier, the mixer, the filters — that chain is generic and gets reused.
What changes is the final stage tuned to the specific band and application. Transfer learning is the
same reuse pattern applied to a learned feature extractor.

| What I build | Why it exists |
|---|---|
| 📥 Fruits-360 dataset | Real images organised into per-class folders |
| 🔄 Augmented data generators | Synthetic variation so the model generalises |
| 🏗️ VGG16 base (frozen) | A pre-trained, general-purpose feature extractor |
| ➕ Custom classification head | The only part that knows about fruit |
| 🎯 Phase 1 training | Head learns while the base stays fixed |
| 🔬 Phase 2 fine-tuning | Top of the base unfreezes at a tiny learning rate |
| 📊 Evaluation + curves | Test accuracy, and whether it over/underfit |
| 🧪 Sample predictions | Qualitative check on individual images |

**The workflow in nine steps:**

| Step | What happens |
|---|---|
| 1 | Import libraries, set dataset paths |
| 2 | Build data generators with augmentation on train only |
| 3 | Load VGG16 without its top, freeze it, attach a custom head |
| 4 | Compile with categorical cross-entropy + Adam |
| 5 | Train the head with early stopping and LR scheduling |
| 6 | Unfreeze the last 5 VGG16 layers, re-compile, fine-tune |
| 7 | Evaluate on the held-out test set |
| 8 | Plot accuracy and loss curves |
| 9 | Predict on individual sample images and inspect |

## 🧩 Theory

### 🏗️ What transfer learning actually reuses

A convolutional network learns a hierarchy. Early layers respond to primitive patterns; deeper layers
compose those into progressively more abstract structure:

```
input image
    │
    ▼
┌──────────────┐  edges, colour blobs, gradients      ─┐
│ conv block 1 │                                       │
├──────────────┤  corners, simple textures             │  GENERIC
│ conv block 2 │                                       │  transfers to
├──────────────┤  repeating patterns, motifs           │  almost any
│ conv block 3 │                                       │  vision task
├──────────────┤  object parts                        ─┘
│ conv block 4 │
├──────────────┤  whole-object structure              ─┐  SPECIFIC
│ conv block 5 │                                       │  to ImageNet's
├──────────────┤                                       │  1000 classes
│ dense head   │  "this is a golden retriever"        ─┘  ← I discard this
└──────────────┘
```

`include_top=False` throws away that final classifier. What remains is a feature extractor: it turns
an image into a rich numerical description, and that description is useful whether the image contains
a dog or a pear.

This is a filter bank I didn't have to design. The characterisation work — millions of images, weeks
of GPU time — was done once by someone else, and I inherit it.

### ❄️ Frozen vs trainable

Setting `layer.trainable = False` excludes a layer's weights from gradient updates. It still runs a
forward pass; it just doesn't learn.

Freezing the base matters for a concrete reason. My new head starts with **random** weights, so the
first few batches produce large, essentially meaningless gradients. If the base were trainable, those
garbage gradients would flow straight back into carefully-tuned ImageNet weights and destroy them.
Freezing protects the asset while the head finds its footing.

### 🔬 The two-phase recipe

This is why the notebook trains twice:

| Phase | Base | Head | Learning rate | Purpose |
|---|---|---|---|---|
| **1 — Head training** | ❄️ Frozen | 🔥 Trainable | Adam default (0.001) | Get the head to something sensible without risking the base |
| **2 — Fine-tuning** | 🔥 Last 5 layers | 🔥 Trainable | 0.00001 (100× smaller) | Nudge high-level features toward fruit specifics |

The tiny learning rate in phase 2 is the whole point. I'm not retraining those layers, I'm *trimming*
them — the same distinction as calibrating a instrument that's already close versus rebuilding it.
A large step here would undo the pre-training I'm trying to exploit.

### 🔢 Categorical cross-entropy

For $C$ classes, the model outputs a probability distribution via softmax:

$$p_i = \frac{e^{z_i}}{\sum_{j=1}^{C} e^{z_j}}$$

Where $z_i$ is the raw score (logit) for class $i$. Softmax exponentiates and normalises, so outputs
are positive and sum to 1 — a proper distribution over classes. It's normalising power across
channels so the total is fixed and only the *relative* distribution carries information.

The loss compares that distribution against the one-hot true label $y$:

$$\mathcal{L} = -\sum_{i=1}^{C} y_i \log(p_i)$$

Because $y$ is one-hot, all terms vanish except the true class, so this reduces to:

$$\mathcal{L} = -\log(p_{\text{true}})$$

The loss is the negative log of the probability assigned to the correct answer. Predict the truth with
probability 1 and loss is 0; predict it with probability 0.01 and loss is 4.6. The log makes confident
mistakes extremely expensive — which is exactly the pressure you want.

### 🔄 Data augmentation

Augmentation applies random transformations to each training image, so the model never sees quite the
same picture twice:

| Transform | Setting | What it simulates |
|---|---|---|
| `rotation_range=20` | ±20° | Camera not perfectly level |
| `width_shift_range=0.1` | ±10% | Subject off-centre horizontally |
| `height_shift_range=0.2` | ±20% | Subject off-centre vertically |
| `shear_range=0.2` | Slant | Off-axis viewing angle |
| `zoom_range=0.2` | ±20% | Varying distance to subject |
| `horizontal_flip=True` | Mirror | Object orientation is arbitrary |
| `fill_mode='nearest'` | Edge fill | How to fill pixels exposed by a transform |

This is the same discipline as testing a receiver against deliberately impaired signals — multipath,
Doppler shift, fading — instead of only on a clean bench signal. A receiver validated only on ideal
input fails in the field. A classifier trained only on perfectly framed images fails on real photos.

⚠️ **Augmentation applies to training data only.** Validation and test sets get rescaling and nothing
else. Augmenting them would mean measuring performance on distorted images, which tells me nothing
about real-world accuracy.

### 📉 Learning rate scheduling

`ReduceLROnPlateau` cuts the learning rate when validation loss stops improving:

$$\eta \leftarrow \max(\eta_{\min},\; \eta \cdot \text{factor}), \qquad \text{factor} = 0.2$$

Large steps make fast progress early but overshoot near a minimum. Cutting the step size on a plateau
lets the optimiser settle into detail it was previously stepping over — a coarse sweep to find the
region, then a fine sweep to locate the exact point.

| Callback | Trigger | Action |
|---|---|---|
| `ReduceLROnPlateau` | `val_loss` flat for 2 epochs | Multiply LR by 0.2 (floor 1e-6) |
| `EarlyStopping` | `val_loss` flat for 5 epochs | Stop, restore best weights |

## Part 1 — 📥 Getting the Dataset

First I suppress the noisier Keras warnings so the output stays readable, then install pinned library
versions. Pinning matters here — TensorFlow, NumPy and SciPy have real cross-version incompatibilities,
and an unpinned environment can fail in confusing ways.

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="keras.src.trainers.data_adapters.py_dataset_adapter")
warnings.filterwarnings("ignore", category=UserWarning, module="keras.src.trainers.epoch_iterator")

import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # Suppress all warnings and info messages

In [ ]:
!pip install tensorflow==2.16.2
!pip install matplotlib==3.9.2
!pip install numpy==1.26.4
!pip install scipy==1.14.1
!pip install scikit-learn==1.5.2

### 🗂️ The directory structure Keras expects

`flow_from_directory` infers class labels **from folder names**. There is no separate label file — the
directory layout *is* the annotation:

```
dataset/
├── train/
│   ├── Class1/
│   ├── Class2/
│   ├── Class3/
│   └── (other classes...)
├── val/
│   ├── Class1/
│   ├── Class2/
│   ├── Class3/
│   └── (other classes...)
└── test/
    ├── Class1/
    ├── Class2/
    ├── Class3/
    └── (other classes...)
```

Class indices are assigned by sorting folder names **alphabetically**, which is worth remembering:
`apple_braeburn_1` becomes index 0 not because it's special, but because it sorts first. I'll need
that mapping in reverse at prediction time.

Now the download. The archive is large, so it's extracted in batches of 2000 files rather than all at
once — unzipping tens of thousands of files in a single call can exhaust memory in a constrained
environment.

In [ ]:
import os
import urllib.request
import zipfile

# Define dataset URL and paths
url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/4yIRGlIpNfKEGJYMhZV52g/fruits-360-original-size.zip"
local_zip = "fruits-360-original-size.zip"
extract_dir = "fruits-360-original-size"

def download_dataset(url, output_file):
    """Download the dataset using urllib from the Python standard library.

    The original used `subprocess.run(["wget", ...])`, which fails on macOS
    (and most Windows setups) because wget isn't installed by default.
    urllib ships with Python, so this works everywhere.

    Downloads to a temporary `.part` file and only renames on success, so an
    interrupted download can't leave a corrupt archive that the existence
    check below would happily skip over.
    """
    print("Downloading the dataset...")
    tmp_file = output_file + ".part"

    def _progress(block_num, block_size, total_size):
        if total_size > 0:
            downloaded = block_num * block_size
            pct = min(100.0, downloaded * 100 / total_size)
            print(f"\r  {pct:5.1f}%  ({downloaded / 1e6:.0f} / {total_size / 1e6:.0f} MB)",
                  end="", flush=True)

    try:
        urllib.request.urlretrieve(url, tmp_file, reporthook=_progress)
        os.replace(tmp_file, output_file)   # atomic rename — only on success
        print("\nDownload completed.")
    except Exception:
        if os.path.exists(tmp_file):
            os.remove(tmp_file)             # don't leave a half-file behind
        raise

def extract_zip_in_chunks(zip_file, extract_to, batch_size=2000):
    """
    Extract a large zip file in chunks to avoid memory bottlenecks.
    Processes a specified number of files (batch_size) at a time.
    """
    print("Extracting the dataset in chunks...")
    os.makedirs(extract_to, exist_ok=True)  # Ensure the extraction directory exists

    with zipfile.ZipFile(zip_file, 'r') as zip_ref:
        files = zip_ref.namelist()  # List all files in the archive
        total_files = len(files)

        for i in range(0, total_files, batch_size):
            batch = files[i:i+batch_size]
            for file in batch:
                zip_ref.extract(file, extract_to)  # Extract each file in the batch
            print(f"Extracted {min(i+batch_size, total_files)} of {total_files} files...")

    print(f"Dataset successfully extracted to '{extract_to}'.")

# Main script execution
if __name__ == "__main__":
    # Download the dataset if not already downloaded
    if not os.path.exists(local_zip) and not os.path.exists(extract_dir):
        download_dataset(url, local_zip)
    else:
        print("Dataset already downloaded or extracted.")

    # Extract the dataset if not already extracted
    if not os.path.exists(extract_dir):
        extract_zip_in_chunks(local_zip, extract_dir)
    else:
        print("Dataset already extracted.")

    # Optional cleanup of the zip file
    if os.path.exists(local_zip):
        os.remove(local_zip)
        print(f"Cleaned up zip file: {local_zip}")

📝 **Notes on this cell:**

- **Downloading with `urllib`, not `wget`.** The source material shelled out to `wget` via
  `subprocess.run(["wget", ...])`. That fails immediately on macOS with
  `FileNotFoundError: [Errno 2] No such file or directory: 'wget'` — macOS doesn't ship wget, and
  neither does a default Windows install. `urllib.request` is part of the Python standard library, so
  it works on any machine that can run the notebook. One less environment assumption.
- **Atomic download.** The file lands as `.part` and is only renamed on success. Without this, an
  interrupted download leaves a truncated `.zip` on disk, and the `os.path.exists` guard below would
  cheerfully skip re-downloading it — producing a confusing `BadZipFile` error at extraction time
  instead of an obvious network error.
- **Idempotent.** The existence guards mean re-running won't re-download several hundred MB. The
  download check also looks for the *extracted* directory, since the cell deletes the zip when it's
  done — otherwise a second run would re-download an archive it no longer needs.
- The download can take a while depending on connection speed.
- If warnings about CUDA or cuDNN appear, the system is running on CPU. Everything still works,
  training is just slower.

## Part 2 — 🔢 Imports and Dataset Paths

Everything needed for the pipeline, plus the three directory paths.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Dropout, BatchNormalization
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping

# Set dataset paths
train_dir = 'fruits-360-original-size/fruits-360-original-size/Training'
val_dir = 'fruits-360-original-size/fruits-360-original-size/Validation'
test_dir = 'fruits-360-original-size/fruits-360-original-size/Test'

📝 **What each import does:**

| Import | Role |
|---|---|
| `ImageDataGenerator` | Loads images from disk in batches and applies augmentation |
| `VGG16` | The pre-trained network I'm borrowing |
| `Sequential` | Container for a linear stack of layers |
| `Dense` | Fully connected layer — the classification head |
| `Dropout` | Regularisation: randomly zeroes activations during training |
| `BatchNormalization` | Normalises activations, stabilises and speeds up training |
| `ReduceLROnPlateau` | Cuts learning rate when progress stalls |
| `EarlyStopping` | Halts training when validation loss stops improving |

⚠️ `Flatten` is imported here but never used — the model uses `GlobalAveragePooling2D` instead
(Part 4 explains why). Harmless, but it's a leftover from a different architecture and I'm noting it
rather than letting it quietly imply the model flattens.

## Part 3 — 🔄 Data Generators and Augmentation

Three generators, but only one augments. Training data gets the full random-transform treatment;
validation and test get rescaling alone.

**On rescaling.** `rescale=1.0/255.0` maps pixel values from $[0, 255]$ to $[0, 1]$:

$$x_{\text{scaled}} = \frac{x}{255}$$

Neural networks train poorly on large-magnitude inputs — activations saturate and gradients behave
badly. Normalising the input range is the same instinct as keeping a signal within an ADC's linear
region instead of driving it into clipping.

In [ ]:
# Image data generators
train_datagen = ImageDataGenerator(
    rescale=1.0/255.0,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator(rescale=1.0/255.0)
test_datagen = ImageDataGenerator(rescale=1.0/255.0)

# Load images from directories
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(64, 64),
    batch_size=16,
    class_mode='categorical'
)

val_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=(64, 64),
    batch_size=16,
    class_mode='categorical'
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(64, 64),
    batch_size=16,
    class_mode='categorical'
)

📝 **Key parameters:**

| Parameter | Value | Meaning |
|---|---|---|
| `target_size` | `(64, 64)` | Every image resized to 64×64 pixels |
| `batch_size` | 16 | Images per gradient step |
| `class_mode` | `'categorical'` | One-hot encoded labels for multi-class |

The output line `Found N images belonging to C classes` is the confirmation that the folder structure
was parsed correctly. If `C` looks wrong, the directory layout is wrong.

⚠️ **64×64 is small, and it has a real consequence.** VGG16 was trained at 224×224 and contains five
max-pooling stages, each halving spatial dimensions:

$$64 \div 2^5 = 2$$

So the final convolutional feature map is just **2×2 spatial** (×512 channels). At 224×224 it would be
7×7. I'm discarding a lot of spatial detail, which matters when distinguishing visually similar fruit
varieties. The tradeoff is speed — 64×64 trains roughly 12× faster than 224×224 per image. It's the
right call for a CPU-based run, but it's a ceiling on achievable accuracy, not a free choice.

## Part 4 — 🏗️ Building the VGG16-Based Model

Three moves: load VGG16 without its classifier, freeze it, then stack a new head on top.

In [ ]:
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, BatchNormalization, Dropout

# Load VGG16 with pre-trained weights
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(64, 64, 3))

# Freeze the base model layers
for layer in base_model.layers:
    layer.trainable = False

# Build the model
model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(256, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(train_generator.num_classes, activation='softmax')
])

📝 **Argument by argument:**

| Argument | Effect |
|---|---|
| `weights='imagenet'` | Load the pre-trained weights, not random ones — this is the entire point |
| `include_top=False` | Drop the 1000-class ImageNet classifier |
| `input_shape=(64, 64, 3)` | 64×64 RGB; must match `target_size` above |
| `layer.trainable = False` | Freeze every base layer for phase 1 |

**The head, layer by layer:**

| Layer | Why it's there |
|---|---|
| `GlobalAveragePooling2D()` | Collapses each 2×2 feature map to a single average → 512 numbers |
| `Dense(256, relu)` | Learns fruit-specific combinations of those features |
| `BatchNormalization()` | Normalises activations; stabilises and accelerates training |
| `Dropout(0.3)` | Randomly zeroes 30% of activations each step — prevents co-adaptation |
| `Dense(num_classes, softmax)` | One output per class, normalised to a probability distribution |

🔍 **Why `GlobalAveragePooling2D` and not `Flatten`.** Both reduce the 3D feature map to a vector, but
very differently:

| | Output size (2×2×512) | Parameters into `Dense(256)` |
|---|---|---|
| `Flatten()` | 2048 | 2048 × 256 ≈ **524k** |
| `GlobalAveragePooling2D()` | 512 | 512 × 256 ≈ **131k** |

Global average pooling takes the mean over each channel's spatial extent — averaging a measurement
over an observation window instead of keeping every individual sample. Four times fewer parameters,
much less prone to overfitting, and it makes the head insensitive to *where* in the frame a feature
appeared. For classification, presence matters more than position.

⚠️ **Two things I corrected from the source material:**

1. The original imported `MobileNetV2` in this cell and never used it. I removed the import — a
   dangling reference to a different architecture in a VGG16 notebook is actively misleading.
2. The original explanation described this head as "Flatten the output, then add dense layers." The
   code does not flatten; it global-average-pools. I've documented what the code actually does.

`train_generator.num_classes` reads the class count straight from the directory scan, so the output
layer sizes itself automatically. No hardcoded number to fall out of sync.

## Part 5 — ⚙️ Compiling the Model

Three choices that have to agree with each other and with the data.

In [ ]:
model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

📝 **Why each choice:**

| Setting | Value | Reason |
|---|---|---|
| `loss` | `categorical_crossentropy` | Multi-class, one-hot labels — pairs with softmax output |
| `optimizer` | `adam` | Adaptive per-parameter learning rates; strong default |
| `metrics` | `accuracy` | Human-readable score; not what's optimised, just what's reported |

⚠️ **Loss and label format must match.** `categorical_crossentropy` expects one-hot labels, which is
exactly what `class_mode='categorical'` produces. Had I used `class_mode='sparse'` (integer labels),
this would need to be `sparse_categorical_crossentropy`. Mismatching them is a common and confusing
error, because it often fails at shape-check time rather than saying anything about labels.

**Loss vs metric.** Gradient descent optimises the *loss*; accuracy is only displayed. They aren't the
same thing — a model can improve loss (getting more confident on things it already had right) while
accuracy stays flat.

## Part 6 — 🎯 Phase 1: Training the Classification Head

The base is frozen, so this trains only the head — roughly 131k + 65k parameters instead of VGG16's
~14.7 million. Fast, and the pre-trained weights are in no danger.

In [ ]:
import tensorflow as tf
from tensorflow.keras.mixed_precision import set_global_policy

# Define callbacks
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
lr_scheduler = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=2, min_lr=1e-6, verbose=1)
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# Set the global float policy explicitly (see note below — this is NOT mixed precision)
set_global_policy('float32')

steps_per_epoch = 50
validation_steps = 25

history = model.fit(
    train_generator,
    epochs=5,
    validation_data=val_generator,
    steps_per_epoch=steps_per_epoch,
    validation_steps=validation_steps,
    callbacks=[lr_scheduler, early_stopping]
)

📝 **The callbacks:**

| Callback | Watches | Patience | Action |
|---|---|---|---|
| `ReduceLROnPlateau` | `val_loss` | 2 epochs | LR × 0.2, floor 1e-6, prints when it fires |
| `EarlyStopping` | `val_loss` | 5 epochs | Stop and restore best weights |

`restore_best_weights=True` is the important flag. Without it, training ends holding whatever weights
the *last* epoch produced — which by definition were the ones that failed to improve. With it, the
model rolls back to its best checkpoint.

⚠️ **Three limitations worth naming honestly:**

1. **The comment about mixed precision was wrong, so I fixed it.** The original said
   `# Enable mixed precision (if on GPU)` above `set_global_policy('float32')`. Float32 is the
   *default* — it's the opposite of mixed precision. Real mixed precision would be
   `set_global_policy('mixed_float16')`. The line is harmless (it makes the default explicit) but the
   comment claimed the opposite of what the code did.

2. **`EarlyStopping(patience=5)` with `epochs=5` can never fire.** It needs 5 epochs of *no improvement*
   after a best epoch, so it would need at least 6 epochs to trigger. As configured it's inert. The
   mechanism is correct; the budget doesn't reach it.

3. **`steps_per_epoch=50` with `batch_size=16` means 800 images per epoch** — a small fraction of
   Fruits-360. Five epochs is a demonstration, not convergence. Expect modest accuracy.

$$\text{images per epoch} = 50 \times 16 = 800$$

## Part 7 — 🔬 Phase 2: Fine-Tuning the Top of VGG16

Now I unfreeze the **last 5 layers** of VGG16 and continue training at a much lower learning rate.

The reasoning follows the hierarchy from the Theory section. Early layers detect edges and textures —
those are already correct for fruit and shouldn't move. The last few layers encode high-level,
ImageNet-specific structure, and *that's* what benefits from adapting to fruit.

**Why only 5 layers, and why such a small learning rate:**

| Choice | Consequence |
|---|---|
| Unfreeze last 5 only | Fewer trainable parameters → less overfitting on a modest dataset, faster |
| Unfreeze everything | Risks destroying general features; needs far more data and time |
| LR = 1e-5 | Small nudges — preserves pre-training while allowing adaptation |
| LR = 1e-3 (Adam default) | Large steps would wreck exactly the weights I'm trying to reuse |

The learning rate is the critical parameter here. This is a **fine trim of an already-calibrated
instrument**, not a recalibration from scratch.

In [ ]:
# Import necessary libraries
import tensorflow as tf  # Import TensorFlow for accessing tf.keras
from tensorflow.keras.optimizers import Adam

# Check the number of layers in the base model
num_layers = len(base_model.layers)
print(f"The base model has {num_layers} layers.")

# Unfreeze the last 5 layers for fine-tuning
for layer in base_model.layers[-5:]:
    layer.trainable = True

# Freeze BatchNorm layers (no-op for VGG16, which has none — see note below)
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

# Re-compile the model — REQUIRED after changing any layer's trainable flag.
# Learning rate is 100x SMALLER than Adam's default, to avoid destroying pre-trained weights.
model.compile(
    loss='categorical_crossentropy',
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    metrics=['accuracy']
)

# Continue training with the same step budget
history_fine = model.fit(
    train_generator,
    epochs=5,
    validation_data=val_generator,
    steps_per_epoch=steps_per_epoch,
    validation_steps=validation_steps,
    callbacks=[lr_scheduler, early_stopping]
)

📝 **`model.compile()` must be called again — this is not optional.** Keras builds its list of
trainable weights at compile time. Changing `layer.trainable` after compiling has *no effect* until
recompilation. Skip this and the fine-tuning silently does nothing: it runs, prints plausible numbers,
and the base never updates. A failure that looks exactly like success is the worst kind.

⚠️ **Three corrections from the source material:**

1. **The comment said "Higher learning rate for faster convergence."** It's 1e-5, which is **100×
   lower** than Adam's 1e-3 default. The value is right and the intent is right; the comment described
   the opposite. Fixed.

2. **The explanation referenced `RMSprop(learning_rate=1e-5)`** while the code uses `Adam`. I've kept
   Adam (what the code actually does) and removed the contradictory claim.

3. **The BatchNorm-freezing loop is a no-op on VGG16.** VGG16 predates batch normalisation and contains
   none — the `isinstance` check never matches. It's harmless boilerplate carried over from a
   ResNet/MobileNet recipe, where it genuinely matters (BN layers update running statistics during
   fine-tuning and can destabilise a small-batch run). I've kept the code so the pattern is visible,
   but relabelled the comment so it doesn't imply something is happening that isn't.

`print(f"The base model has {num_layers} layers.")` should report **19** for VGG16 without its top.

## Part 8 — 📊 Evaluating on the Test Set

The test set is data the model has never seen — not during training, and not for early-stopping
decisions. Validation loss influenced when training stopped and when the LR dropped, so validation
accuracy is mildly optimistic. Test accuracy is the honest number.

In [ ]:
# Evaluate on the test set
test_loss, test_accuracy = model.evaluate(test_generator, steps=50)
print(f"Test Accuracy: {test_accuracy:.2f}")

📝 **The three-way split and why it exists:**

| Split | Used for | Model sees it |
|---|---|---|
| **Train** | Gradient updates | Every epoch, augmented |
| **Validation** | Early stopping, LR scheduling | Every epoch, for monitoring only |
| **Test** | Final unbiased estimate | Once, at the very end |

Validation data never contributes gradients, but it *does* influence the model indirectly through
those callback decisions. That's a subtle form of leakage, which is precisely why a third, completely
untouched split exists.

`steps=50` evaluates 50 × 16 = 800 test images rather than the full set — a sample, so the number
carries sampling noise. Dropping `steps` entirely would evaluate everything.

## Part 9 — 📈 Visualising Training Performance

Curves reveal things a single accuracy number can't. All four traces go on one axis — phase 1 and
phase 2, training and validation.

In [ ]:
# Plot accuracy and loss curves
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.plot(history_fine.history['accuracy'], label='Fine-tuned Training Accuracy')
plt.plot(history_fine.history['val_accuracy'], label='Fine-tuned Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.title('Training and Validation Accuracy')
plt.grid(True)
plt.show()

plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.plot(history_fine.history['loss'], label='Fine-tuned Training Loss')
plt.plot(history_fine.history['val_loss'], label='Fine-tuned Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.title('Training and Validation Loss')
plt.grid(True)
plt.show()

📝 **How to read these curves:**

| Pattern | Diagnosis | Response |
|---|---|---|
| Both accuracies rise together | 🟢 Learning well | Train longer |
| Training ↑, validation flat or ↓ | 🔴 [[overfitting]] | More dropout, more augmentation, more data |
| Both flat and low | 🟡 [[underfitting]] | Bigger head, higher LR, unfreeze more layers |
| Validation noisy but trending up | 🟡 Small validation set | More `validation_steps` |

The training/validation **gap** is the signal that matters most. A widening gap means the model is
memorising rather than generalising.

⚠️ **The x-axis is misleading as plotted.** Both `history` and `history_fine` start their epoch
indexing at 0, so phase 2's curves are drawn on top of phase 1's instead of continuing after them.
Visually it suggests two parallel runs rather than one sequential process. Fixing it means offsetting
the second range:

```python
n = len(history.history['accuracy'])
plt.plot(range(n), history.history['accuracy'], label='Phase 1 — training')
plt.plot(range(n, n + len(history_fine.history['accuracy'])),
         history_fine.history['accuracy'], label='Phase 2 — fine-tuned')
```

That's Sandbox exercise 3. I've kept the original plotting code here so the source behaviour is
preserved and the flaw is visible rather than silently patched.

## Part 10 — 🧪 Predictions on Sample Images

Aggregate accuracy hides the interesting failures. Here I run individual images through the model and
display each with its true and predicted label.

The one piece of real machinery is the **index-to-name lookup**. `train_generator.class_indices` maps
`name → index`, but a prediction gives me an index, so I need the inverse.

In [ ]:
import os
import numpy as np
from collections import Counter
from tensorflow.keras.preprocessing.image import img_to_array, load_img
import matplotlib.pyplot as plt

# Initialize counters for actual and predicted classes
actual_count = Counter()
predicted_count = Counter()

# Function to get class name from predicted index
def get_class_name_from_index(predicted_index, class_index_mapping):
    """Convert predicted index to class name."""
    for class_name, index in class_index_mapping.items():
        if index == predicted_index:
            return class_name
    return "Unknown"  # Default if index is not found

# Define the function for visualization
def visualize_prediction_with_actual(img_path, class_index_mapping):
    # Extract the true label dynamically from the directory structure
    class_name = os.path.basename(os.path.dirname(img_path))  # Extract folder name (class)

    # Load and preprocess the image
    img = load_img(img_path, target_size=(64, 64))
    img_array = img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    # Predict the class
    prediction = model.predict(img_array)
    predicted_index = np.argmax(prediction, axis=-1)[0]
    predicted_class_name = get_class_name_from_index(predicted_index, class_index_mapping)

    # Update the counters
    actual_count[class_name] += 1
    predicted_count[predicted_class_name] += 1

    # Visualize the image with predictions
    plt.figure(figsize=(2, 2), dpi=100)
    plt.imshow(img)
    plt.title(f"Actual: {class_name}, Predicted: {predicted_class_name}")
    plt.axis('off')
    plt.show()

# Retrieve class index mapping from the training generator
class_index_mapping = train_generator.class_indices
print("Class Index Mapping:", class_index_mapping)  # Debugging: Check the mapping

# Define a list of image paths without hardcoded labels
sample_images = [
    'fruits-360-original-size/fruits-360-original-size/Test/apple_braeburn_1/r0_11.jpg',
    'fruits-360-original-size/fruits-360-original-size/Test/pear_1/r0_103.jpg',
    'fruits-360-original-size/fruits-360-original-size/Test/cucumber_3/r0_103.jpg',
]

# Run the predictions and visualization
for img_path in sample_images:
    visualize_prediction_with_actual(img_path, class_index_mapping)

# Summarise what was actually seen vs predicted
print("\nActual class counts   :", dict(actual_count))
print("Predicted class counts:", dict(predicted_count))

📝 **The preprocessing must match training exactly.** Three steps, in order:

| Step | Code | Why |
|---|---|---|
| Resize | `load_img(..., target_size=(64, 64))` | Model input shape is fixed |
| Rescale | `img_to_array(img) / 255.0` | Same $[0,1]$ normalisation the generator applied |
| Add batch axis | `np.expand_dims(img_array, axis=0)` | `(64,64,3)` → `(1,64,64,3)`; Keras always expects a batch |

Getting the rescale wrong is the classic silent bug here. Feed unscaled $[0,255]$ pixels to a model
trained on $[0,1]$ and it produces confident nonsense — no error, just wrong answers. Inference
preprocessing must mirror training preprocessing exactly.

Reading the true label from the *directory name* rather than hardcoding it is a small but good habit:
the labels can't drift out of sync with the files.

📝 **One addition of mine:** the original populated `actual_count` and `predicted_count` and then never
printed them — dead code. I've added the two summary lines at the end so the counters actually do
something.

🔍 **When a prediction is wrong, the usual suspects:**

| Cause | Explanation |
|---|---|
| **Class similarity** | Apple varieties differ subtly; at 64×64 that detail may simply not survive |
| **Insufficient data** | 800 images/epoch across many classes is thin coverage |
| **Limited fine-tuning** | 5 unfrozen layers, 5 epochs — limited chance to learn fruit-specific features |
| **Aggressive augmentation** | 20° rotation + 20% shear + 20% zoom can distort exactly the cues that distinguish varieties |
| **Resolution ceiling** | The 2×2 final feature map is a hard limit on spatial detail |

Note the third sample is a **cucumber** — Fruits-360 includes vegetables, and it's a reasonable test of
whether the model has learned anything beyond "roundish and colourful."

## 📊 Summary

I built a fruit image classifier by reusing VGG16's ImageNet-trained feature extractor and training
only a small custom head, then fine-tuning the top of the base at a much lower learning rate.

### Pipeline

| Stage | Implementation | Purpose |
|---|---|---|
| 📥 **Data** | Fruits-360, folder-per-class | Labels inferred from directory structure |
| 🔄 **Augmentation** | Rotation, shift, shear, zoom, flip | Synthetic variety → better generalisation |
| 🏗️ **Base** | VGG16, `include_top=False`, frozen | Borrowed general-purpose feature extractor |
| ➕ **Head** | GAP → Dense(256) → BN → Dropout(0.3) → softmax | The only fruit-specific part |
| ⚙️ **Compile** | `categorical_crossentropy` + Adam | Multi-class objective |
| 🎯 **Phase 1** | 5 epochs, base frozen | Head learns without endangering pre-trained weights |
| 🔬 **Phase 2** | 5 epochs, last 5 layers unfrozen, LR 1e-5 | Adapt high-level features to fruit |
| 📊 **Evaluate** | Test set, 50 steps | Unbiased accuracy estimate |

### Key equations

$$p_i = \frac{e^{z_i}}{\sum_j e^{z_j}} \qquad \text{(softmax)}$$

$$\mathcal{L} = -\log(p_{\text{true}}) \qquad \text{(categorical cross-entropy, one-hot)}$$

$$x_{\text{scaled}} = \frac{x}{255} \qquad \text{(rescaling)}$$

$$64 \div 2^5 = 2 \qquad \text{(final spatial resolution after VGG16's 5 pooling stages)}$$

### Design decisions

| Decision | Value | Rationale |
|---|---|---|
| Base network | VGG16 | Simple, well-understood architecture; ImageNet weights |
| Input size | 64×64 | Speed over accuracy — a deliberate, costly tradeoff |
| Pooling | `GlobalAveragePooling2D` | 4× fewer head parameters than `Flatten`; less overfitting |
| Dropout | 0.3 | Moderate regularisation on a small head |
| Fine-tune depth | Last 5 layers | Enough to adapt, few enough to avoid overfitting |
| Fine-tune LR | 1e-5 | 100× below default — trim, don't retrain |

### 🎓 What I take away

1. **Transfer learning is reuse, not a shortcut.** The generic feature extractor was expensive to
   produce and is genuinely general. Only the last stage is task-specific — the same modular reuse as
   a standard RF chain with a retuned final stage.
2. **Freeze first, fine-tune second, and recompile in between.** Changing `trainable` without calling
   `compile()` again silently does nothing. That's a failure mode that looks identical to success.
3. **The fine-tuning learning rate is the critical hyperparameter.** Too high and the pre-trained
   weights — the whole reason for using transfer learning — get destroyed in a few batches.
4. **Augment training data only.** Augmenting validation or test measures performance on distortions
   nobody cares about.
5. **Input resolution sets a hard ceiling.** At 64×64, VGG16's five pooling stages leave a 2×2 feature
   map. No amount of training recovers spatial detail thrown away at the input.

### ⚠️ Gaps to close next

| Gap | Consequence | Fix |
|---|---|---|
| `EarlyStopping(patience=5)` with `epochs=5` | Callback can never fire | Raise epochs above patience, or lower patience |
| `steps_per_epoch=50` (800 images) | Far from convergence | Remove the cap, or raise it substantially |
| 64×64 input | 2×2 final feature map; fine detail lost | 224×224 if compute allows |
| Overlapping x-axis in plots | Phase 2 drawn over phase 1, not after | Offset the second epoch range |
| Test evaluated on 50 steps | Sampling noise in the headline number | Drop `steps` to evaluate the full test set |
| BatchNorm-freezing loop | No-op on VGG16 | Only meaningful on a base that actually contains BN layers |

## 🧪 Sandbox

Space to experiment, roughly in order of expected value:

**1. Raise the training budget.** The single biggest limitation. Remove `steps_per_epoch` so each epoch
sees the whole training set, and raise `epochs` to 20–30 so `EarlyStopping` can actually do its job:

```python
history = model.fit(
    train_generator,
    epochs=30,
    validation_data=val_generator,
    callbacks=[lr_scheduler, early_stopping]   # no steps_per_epoch cap
)
```

**2. Increase input resolution to 224×224.** VGG16's native size — the final feature map becomes 7×7
instead of 2×2. Change `target_size` in all three generators *and* `input_shape` in `VGG16(...)`; they
must match. Much slower, but likely the largest accuracy gain available.

**3. Fix the training-curve x-axis** so phase 2 continues after phase 1 rather than overlapping it:

```python
n = len(history.history['accuracy'])
plt.plot(range(n), history.history['accuracy'], label='Phase 1 — training')
plt.plot(range(n, n + len(history_fine.history['accuracy'])),
         history_fine.history['accuracy'], label='Phase 2 — fine-tuned')
```

**4. Add a confusion matrix.** Far more informative than a single accuracy figure — it shows *which*
classes get confused with which, which is exactly the question raised by similar fruit varieties:

```python
from sklearn.metrics import confusion_matrix, classification_report
preds = model.predict(test_generator)
y_pred = np.argmax(preds, axis=1)
y_true = test_generator.classes
print(classification_report(y_true, y_pred, target_names=list(test_generator.class_indices)))
```

**5. Sweep the fine-tuning depth.** Try unfreezing the last 3, 5, 8, or all 19 layers and compare. At
what point does performance start degrading from overfitting?

**6. Try a different base network.** `MobileNetV2` (far fewer parameters, built for edge deployment),
`ResNet50` (residual connections), or `EfficientNetB0`. All share the same Keras API, so it's close to
a drop-in swap — and MobileNetV2 makes the BatchNorm-freezing loop meaningful, since it actually has
BN layers.

**7. Ablate the augmentation.** Train once with augmentation and once without, same budget. How much
does it actually contribute at this dataset size?

**8. Sweep dropout.** Try 0.0, 0.3, 0.5, 0.7 and watch the train/validation gap in the curves.

**9. Inspect the errors, not the score.** Collect every misclassified test image and display them
grouped by true class. Patterns in the failures are usually more instructive than the aggregate number.

In [ ]:
# 🧪 Sandbox — experiment freely